In [1]:
import pandas as pd

print("SECITON B \n")
orders = pd.DataFrame({
    
'OrderID': [101, 102, 103, 104, 105, 106, 107, 108],
'UnitPrice': [12.5, 45.0, 8.99, 120.0, 19.99, 60.0, 5.5, 89.0],
'Quantity': [3, 1, 10, 2, 5, 1, 20, 4],
'CustomerSince': pd.to_datetime(['2021-02-10','2022-11-05','2020-07-19',
'2023-01-30','2019-09-12','2022-04-23',
'2021-12-01','2023-06-15']),
'OrderDate': pd.to_datetime(['2023-09-01','2023-09-03','2023-09-05',
'2023-09-08','2023-09-10','2023-09-12',
'2023-09-14','2023-09-18']),
'Country': ['India','UK','India','USA','UK','India','USA','India'],
'Returned': [0, 0, 1, 0, 1, 0, 1, 0], # target column
})

print(orders)

SECITON B 

   OrderID  UnitPrice  Quantity CustomerSince  OrderDate Country  Returned
0      101      12.50         3    2021-02-10 2023-09-01   India         0
1      102      45.00         1    2022-11-05 2023-09-03      UK         0
2      103       8.99        10    2020-07-19 2023-09-05   India         1
3      104     120.00         2    2023-01-30 2023-09-08     USA         0
4      105      19.99         5    2019-09-12 2023-09-10      UK         1
5      106      60.00         1    2022-04-23 2023-09-12   India         0
6      107       5.50        20    2021-12-01 2023-09-14     USA         1
7      108      89.00         4    2023-06-15 2023-09-18   India         0


In [ ]:
# B1 - Arithmetic & Ratio Features

print("SECTION B1 \n")
orders['Revenue'] = orders['UnitPrice'] * orders['Quantity']
orders['Price_to_CountryAvg'] = orders['UnitPrice'] / orders.groupby('Country')['UnitPrice'].transform('mean')
print(orders['Revenue'])
print(orders['Price_to_CountryAvg'])

SECTION B1 

0     37.50
1     45.00
2     89.90
3    240.00
4     99.95
5     60.00
6    110.00
7    356.00
Name: Revenue, dtype: float64
0    0.293272
1    1.384828
2    0.210921
3    1.912351
4    0.615172
5    1.407707
6    0.087649
7    2.088099
Name: Price_to_CountryAvg, dtype: float64


In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

print("SECTION B2 \n")

poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_features = poly.fit_transform(orders[['UnitPrice', 'Quantity']])
print("Numerically Identical:", np.allclose(orders['Revenue'], poly_features[:, 2]))

SECTION B2 

Numerically Identical: True


In [ ]:
import numpy as np

print("SECTION B3 \n")
orders['Order_Month'] = orders['OrderDate'].dt.month
orders['Order_DayOfWeek'] = orders['OrderDate'].dt.dayofweek
orders['Order_IsWeekend'] = orders['Order_DayOfWeek'].isin([5, 6]).astype(int)
orders['Customer_Tenure_Days'] = (orders['OrderDate'] - orders['CustomerSince']).dt.days
orders['Order_DayOfWeek_sin'] = np.sin(2 * np.pi * orders['Order_DayOfWeek'] / 7)
orders['Order_DayOfWeek_cos'] = np.cos(2 * np.pi * orders['Order_DayOfWeek'] / 7)

print(orders.head(3))

SECTION B3 

   OrderID  UnitPrice  Quantity CustomerSince  OrderDate Country  Returned  \
0      101      12.50         3    2021-02-10 2023-09-01   India         0   
1      102      45.00         1    2022-11-05 2023-09-03      UK         0   
2      103       8.99        10    2020-07-19 2023-09-05   India         1   

   Revenue  Price_to_CountryAvg  Order_Month  Order_DayOfWeek  \
0     37.5             0.293272            9                4   
1     45.0             1.384828            9                6   
2     89.9             0.210921            9                1   

   Order_IsWeekend  Customer_Tenure_Days  Order_DayOfWeek_sin  \
0                0                   933            -0.433884   
1                1                   302            -0.781831   
2                0                  1143             0.781831   

   Order_DayOfWeek_cos  
0            -0.900969  
1             0.623490  
2             0.623490  


In [ ]:
from sklearn.feature_selection import VarianceThreshold

print("SECTION B4 \n")
orders['Currency'] = ['INR'] * 7 + ['USD']
currency_encoded = pd.get_dummies(orders['Currency'], prefix='Currency', dtype=int)
orders = pd.concat([orders, currency_encoded], axis=1)

numeric_cols = ['UnitPrice', 'Quantity', 'Revenue', 'Price_to_CountryAvg', 
                'Order_Month', 'Order_DayOfWeek', 'Order_IsWeekend', 
                'Customer_Tenure_Days', 'Order_DayOfWeek_sin', 'Order_DayOfWeek_cos', 
                'Currency_INR', 'Currency_USD']

vt = VarianceThreshold(threshold=0.01)
vt.fit(orders[numeric_cols])

features_list = pd.Index(numeric_cols)

kept_vt = features_list[vt.get_support()].tolist()
dropped_vt = features_list[~vt.get_support()].tolist()

print("Kept columns:", kept_vt)
print("Dropped columns:", dropped_vt)

SECTION B4 

Kept columns: ['UnitPrice', 'Quantity', 'Revenue', 'Price_to_CountryAvg', 'Order_DayOfWeek', 'Order_IsWeekend', 'Customer_Tenure_Days', 'Order_DayOfWeek_sin', 'Order_DayOfWeek_cos', 'Currency_INR', 'Currency_USD']
Dropped columns: ['Order_Month']


In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
print("SECTION B5 \n")
skb_cols = ['Revenue', 'UnitPrice', 'Quantity', 'Price_to_CountryAvg', 'Customer_Tenure_Days', 'Order_DayOfWeek_sin', 'Order_DayOfWeek_cos']
skb = SelectKBest(score_func=f_classif, k=4)
skb.fit(orders[skb_cols], orders['Returned'])

scores = pd.Series(skb.scores_, index=skb_cols).sort_values(ascending=False)
print(scores)

SECTION B5 



Quantity                8.165767
Customer_Tenure_Days    6.703542
Price_to_CountryAvg     6.594874
UnitPrice               4.722980
Order_DayOfWeek_sin     0.407282
Revenue                 0.310945
Order_DayOfWeek_cos     0.001588
dtype: float64


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pandas as pd
print("SECTION B6 \n")
skb_cols = ['Revenue', 'UnitPrice', 'Quantity', 'Price_to_CountryAvg', 
            'Customer_Tenure_Days', 'Order_DayOfWeek_sin', 'Order_DayOfWeek_cos']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(orders[skb_cols])

rfe = RFE(estimator=LogisticRegression(), n_features_to_select=4)
rfe.fit(X_scaled, orders['Returned'])
features_list = pd.Index(skb_cols)
rfe_kept = features_list[rfe.get_support()].tolist()

print("RFE Chosen Features:", rfe_kept)

SECTION B6 

RFE Chosen Features: ['UnitPrice', 'Quantity', 'Price_to_CountryAvg', 'Customer_Tenure_Days']


In [ ]:
# SECTION C : DEBUG AND EXPLAIN THE ERROR, WHAT WE CHANGED AND WHY. 

# C1 

from sklearn.preprocessing import PolynomialFeatures
# df has 20 numeric columns
poly = PolynomialFeatures(degree=3, include_bias=False)  # CORRECTION :poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(df)
model.fit(X_poly, y)

# EXPLANATION: Raising 20 features to the 3rd power creates too much data for a computer to handle at once. Lowering the degree to 2 keeps the dataset small, fast, and safe from memory crashes

# C2

df['Enroll_Month'] = df['EnrollDate'].dt.month
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(df[['Enroll_Month', 'Score_Total']], y) 
# CORRECTION : 
# df['Month_Sin'] = np.sin(2 * np.pi * df['Enroll_Month'] / 12)
# df['Month_Cos'] = np.cos(2 * np.pi * df['Enroll_Month'] / 12)
# knn.fit(df[['Month_Sin', 'Month_Cos', 'Score_Total']], y)

# EXPLANATION : Raw month numbers trick distance models into thinking December and January are opposites.

# C3

full_df = pd.concat([X_train, X_test])
corr_matrix = full_df.corr().abs()
# ... drop highly correlated columns using corr_matrix ...
X_train_sel = X_train.drop(columns=to_drop)
X_test_sel = X_test.drop(columns=to_drop)

# CORRECTION :
# corr_matrix = X_train.corr().abs()

# EXPLANATION: Mixing training and testing data together lets secret test patterns leak into your training steps. Checking correlations strictly on the training data keeps your final model test fair and honest.


In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler 
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

np.random.seed(42)
num_records = 100

print("SECTION D \n")
data = {
    'EmployeeID': range(1001, 1001 + num_records),
    'Department': np.random.choice(['Sales', 'R&D', 'HR'], size=num_records),
    'JoinDate': pd.date_range(start='2020-01-01', periods=num_records, freq='D'),
    'MonthlySalary': np.random.uniform(3000, 12000, size=num_records),
    'YearsAtCompany': np.random.randint(1, 10, size=num_records),
    'NumProjects': np.random.randint(1, 7, size=num_records),
    'LastReviewScore': np.random.uniform(1.0, 5.0, size=num_records),
    'Attrition': np.random.choice([0, 1], size=num_records, p=[0.8, 0.2])
}

df=pd.DataFrame(data)
df.to_csv("employee_attribution.csv", index=False)
print("Dataset created successfully")

df['Salary_per_Project'] = df['MonthlySalary'] / df['NumProjects']
df['Projects_per_Year'] = df['NumProjects'] / (df['YearsAtCompany'] + 1)
df['JoinDate'] = pd.to_datetime(df['JoinDate'])
current_date = pd.to_datetime('2026-06-20')  
df['Days_Since_Join'] = (current_date - df['JoinDate']).dt.days

df['Join_Month'] = df['JoinDate'].dt.month
df['Month_Sin'] = np.sin(2 * np.pi * df['Join_Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Join_Month'] / 12)

df = df.drop(columns=['JoinDate'])
 
df = pd.get_dummies(df, columns=['Department'], dtype=int)
X = df.drop(columns=['Attrition'])
y = df['Attrition']

continuous_cols = ['MonthlySalary', 'YearsAtCompany', 'NumProjects', 
                   'LastReviewScore', 'Salary_per_Project', 'Projects_per_Year', 'Days_Since_Join']
scaler = StandardScaler()
X[continuous_cols] = scaler.fit_transform(X[continuous_cols])

selector_var = VarianceThreshold(threshold=0.01)
X_high_var = selector_var.fit_transform(X)
remaining_features = X.columns[selector_var.get_support()]
X_filtered = pd.DataFrame(X_high_var, columns=remaining_features)

selector_kbest = SelectKBest(score_func=f_classif, k=6)
X_best = selector_kbest.fit_transform(X_filtered, y)
final_features = X_filtered.columns[selector_kbest.get_support()]
final_df = pd.DataFrame(X_best, columns=final_features)
final_df['Attrition'] = y.values

print("PIPELINE METRICS")
print(f"Features selected by SelectKBest: {list(final_features)}")

final_df.to_csv("employees_feature_engineered.csv", index=False)

SECTION D 

Dataset created successfully
PIPELINE METRICS
Features selected by SelectKBest: ['EmployeeID', 'MonthlySalary', 'Salary_per_Project', 'Days_Since_Join', 'Join_Month', 'Month_Sin']


In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score 
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

df = pd.read_csv(r"C:\Users\USER\Downloads\bank.csv")

df['Target'] = df['deposit'].str.lower().str.strip().replace({'yes': 1, 'no': 0})
df['Target'] = pd.to_numeric(df['Target'], errors='coerce')
df = df.dropna(subset=['Target'])

raw_numeric_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
X_raw = df[raw_numeric_cols]
y = df['Target']

df['Target'] = df['Target'].astype(int)
y = df['Target']
X_raw = df[raw_numeric_cols]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42) 

scaler_base = StandardScaler()
X_train_raw_scaled = scaler_base.fit_transform(X_train_raw)
X_test_raw_scaled = scaler_base.transform(X_test_raw)

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train_raw_scaled, y_train)

y_pred_base = baseline_model.predict(X_test_raw_scaled)
base_acc = accuracy_score(y_test, y_pred_base)
base_f1 = f1_score(y_test, y_pred_base)

new_df = df.copy()
new_df['Balance_to_Age_Ratio'] = new_df['balance'] / new_df['age']
new_df['Total_Contacts'] = new_df['campaign'] + new_df['previous']
new_df['Balance_x_Duration'] = new_df['balance'] * new_df['duration']

month_map = {'jan':1, 'feb':2, 'mar':3, 'apr':4, 'may':5, 'jun':6, 'jul':7, 'aug':8, 'sep':9, 'oct':10, 'nov':11, 'dec':12}
new_df['month_num'] = new_df['month'].str.lower().map(month_map)
new_df['Month_Sin'] = np.sin(2 * np.pi * new_df['month_num'] / 12)
new_df['Month_Cos'] = np.cos(2 * np.pi * new_df['month_num'] / 12)

new_df = new_df.drop(columns=['deposit', 'target', 'month', 'month_num'], errors = 'ignore')
new_df = pd.get_dummies(new_df, dtype=int)

X_train_new, X_test_new = train_test_split(new_df, test_size=0.2, random_state=42)
scaler_new = StandardScaler()
X_train_new_scaled = pd.DataFrame(scaler_new.fit_transform(X_train_new), columns=new_df.columns)
X_test_new_scaled = pd.DataFrame(scaler_new.transform(X_test_new), columns=new_df.columns)

var_selector = VarianceThreshold(threshold=0.01)
X_train_var = var_selector.fit_transform(X_train_new_scaled)
X_test_var = var_selector.transform(X_test_new_scaled)

passed_var_cols = X_train_new_scaled.columns[var_selector.get_support()]
X_train_var_df = pd.DataFrame(X_train_var, columns=passed_var_cols)
X_test_var_df = pd.DataFrame(X_test_var, columns=passed_var_cols)

kbest_selector = SelectKBest(score_func=f_classif, k=6)
X_train_final = kbest_selector.fit_transform(X_train_var_df, y_train)
X_test_final = kbest_selector.transform(X_test_var_df)
final_selected_cols = passed_var_cols[kbest_selector.get_support()]

final_model = LogisticRegression(max_iter=1000, random_state=42)
final_model.fit(X_train_final, y_train)

y_pred_final = final_model.predict(X_test_final)
final_acc = accuracy_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)

print("FINAL ACCURACY RESULTS")
print("Baseline Accuracy:", base_acc)
print("Baseline F1-Score:", base_f1)

print("NEW FEATURE METRICS \n")
print("New Features Accuracy:", final_acc)
print("New Features F1-Score:", final_f1)

print("BEST FEATURES \n")
print("The Best 6 Features Chosen Are:")
print(list(final_selected_cols))

FINAL ACCURACY RESULTS
Baseline Accuracy: 0.7407075682937752
Baseline F1-Score: 0.7126550868486352
NEW FEATURE METRICS 

New Features Accuracy: 0.7711598746081505
New Features F1-Score: 0.743859649122807
BEST FEATURES 

The Best 6 Features Chosen Are:
['duration', 'Balance_x_Duration', 'contact_cellular', 'contact_unknown', 'poutcome_success', 'poutcome_unknown']
